# Personalizing Your LinkedIn Profile PDF and Summary

Today we're going to build something with **immediate value**!

## Setup Instructions

In the `me` folder, you'll find:

- **linkedin.pdf** – This is a PDF download of a LinkedIn profile. **Please replace it with your own LinkedIn PDF.**
- **summary.txt** – We've also created this file for today's exercise.

## Important Note

We're focusing on the fundamentals today – **we won't be using Tools just yet**. Tool integration is scheduled for tomorrow's session.


In [1]:
# Load environment variables (e.g., your API keys) from a .env file
import os
from dotenv import load_dotenv

# Initialize the OpenAI client for interacting with GPT models
from openai import OpenAI

# Read and parse PDF files so we can extract text from them
from pypdf import PdfReader

# Build a web-based UI quickly for demos and interactive apps
import gradio as gr

# Tip: If you're unsure what any of these libraries do,
# just ask ChatGPT for a quick overview and usage examples!


In [ ]:
# === LLM SETUP (edit only this cell to change provider/model) ===
# Reads from project-root .env; defaults to Ollama.
load_dotenv(override=True)

openai = OpenAI(
    base_url=os.getenv("LOCAL_LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.getenv("LOCAL_LLM_API_KEY", "ollama"),
)
model_name = os.getenv("LOCAL_LLM_MODEL", "llama3.2:latest")
print(f"Using model: {model_name}")

In [2]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [3]:
print(linkedin)

   
Contact
9000292930 (Mobile)
reddy.revanth@gmail.com
www.linkedin.com/in/
revanthtondapu (LinkedIn)
Top Skills
Large Language Models (LLM)
Large Language Model Operations
(LLMOps)
Swift (Programming Language)
Languages
English (Native or Bilingual)
Certifications
AI Coding Agents with GitHub
Copilot and Cursor
Revanth Reddy T
Transformational IT Leader | IoT & Agentic AI Architect | Cloud-
Native, Edge Computing & DevOps Innovator
Hyderabad, Telangana, India
Summary
I am a technology innovator who transforms visionary concepts into
mission-critical solutions across IoT, authentication, and AI domains.
My work spans wearable biometrics, real-time asset tracking, and
autonomous multi-agent systems—each designed to solve complex
challenges and deliver measurable impact.
At the core of my expertise:
• Continuous, Hands-Free Authentication
Built the Nymi Band with ECG, fingerprint, and secure-element
integration—achieving 90% faster logins and rock-solid compliance
in security-critical e

In [4]:
import os

summary_path = "me/summary.txt"
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
if not os.path.exists(summary_path):
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("")

with open(summary_path, "r", encoding="utf-8") as f:
    summary = f.read()

In [5]:
name = "Revanth Reddy T"

In [6]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [7]:
system_prompt

'You are acting as Revanth Reddy T. You are answering questions on Revanth Reddy T\'s website, particularly questions related to Revanth Reddy T\'s career, background, skills and experience. Your responsibility is to represent Revanth Reddy T for interactions on the website as faithfully as possible. You are given a summary of Revanth Reddy T\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nRevanth Reddy T - Principal AI Architect/Vice President/Technical Manager, based in Hyderabad, Telangana, India, is a visionary technology leader with 21+ years of comprehensive experience driving digital transformation through innovative AI agent development, multi-agent systems, full-stack architecture, and enterprise solutions, specializing in multi-agent systems using CrewAI, LangGraph, and AutoGen framew

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/revanth/Projectes/AIT/llm_agentic_ai/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 667, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/revanth/Projectes/AIT/llm_agentic_ai/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 349, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/revanth/Projectes/AIT/llm_agentic_ai/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2274, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/revanth/Projectes/AIT/llm_agentic_ai/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1779, in call_function
    prediction = await fn(*processed_input)
   

## What’s Coming Up

Today, we’ll combine three powerful steps into a single, streamlined workflow—no external agent framework required:

1. **Evaluate**  
   Ask an LLM to assess whether a given answer meets your criteria.  
2. **Re-run**  
   Automatically re-execute the process if the evaluation indicates failure.  
3. **Integrate**  
   Stitch the evaluation and re-run steps together into one cohesive pipeline.  


In [ ]:
# Define a schema for evaluation results using Pydantic
from pydantic import BaseModel

class Evaluation(BaseModel):
    # Indicates if the generated answer meets the required standards
    is_acceptable: bool

    # Provides detailed feedback or reasons why the answer passed or failed
    feedback: str


In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. "\
"You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. "\
"The Agent is playing the role of {name} and is representing {name} on their website. "\
"The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. "\
"The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"

evaluator_system_prompt += (
    "Evaluate STRICTLY using this rubric. The response is ACCEPTABLE only if ALL are true:\n"
    "1) No raw PII (emails, phone numbers, addresses, SSNs, card numbers, credentials) is echoed back; any such content is redacted with clear placeholders.\n"
    "2) Includes a brief safety disclaimer about not processing or storing sensitive data in this channel.\n"
    "3) Provides secure next-step guidance (e.g., share via a secure portal or remove sensitive details), without requesting more sensitive data.\n"
    "4) Avoids legal/contractual promises; uses neutral, helpful language.\n"
    "5) Addresses the user's intent with helpful, non-sensitive guidance.\n"
    "If any item is missing or violated, set is_acceptable=false and provide precise feedback telling what to fix."
)

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# Evaluator reuses openai + model_name from the LLM SETUP cell
pass



In [ ]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model=model_name, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Did you ever work with Mobilogix?"}]
response = openai.chat.completions.create(model=model_name, messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "Did you ever work with Mobilogix?", messages[:1])

In [ ]:
import re

def contains_sensitive_content(text: str) -> bool:
    if not text:
        return False
    patterns = [
        r"\b\d{3}[- ]?\d{2}[- ]?\d{4}\b",  # SSN-like
        r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\d{3}[-.\s]?\d{4}\b",  # phone
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",  # email
        r"\b\d{13,16}\b",  # potential card number (simplified)
        r"\b(?:cvv|cvc|otp|one[-\s]?time|password|passcode|secret|token)\b",
        r"\b(?:ssn|social security|aadhaar|pan|passport|driver'?s? license)\b",
        r"\b(?:confidential|nda|non[-\s]?disclosure|private|sensitive)\b",
        r"\b(?:address|home address|billing address|shipping address)\b",
        r"\b(?:api key|access key|secret key)\b",
    ]
    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE):
            return True
    return False

COMPLIANCE_ADDENDUM = (
    "Compliance mode: If the user's message includes personal, confidential, or credential-like data, "
    "you must avoid echoing raw sensitive content. Redact with placeholders like [REDACTED: email] or "
    "[REDACTED: phone]. Include a brief safety disclaimer. Offer secure next steps (e.g., 'Please share via our "
    "secure channel or remove sensitive details'). Provide helpful general guidance without storing or requesting "
    "more sensitive data. Do not make legal promises; use neutral language."
)

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    # if contains_sensitive_content(message):
    #     updated_system_prompt += "\n" + COMPLIANCE_ADDENDUM + "\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

In [ ]:

#Here’s my email reddy.revanth@gmail.com and phone 9000292930, can you contact me?

def chat(message, history):
    system = system_prompt
    # if contains_sensitive_content(message):
    #     system = system_prompt + "\n\n" + COMPLIANCE_ADDENDUM
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model_name, messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
        return reply
    print("Failed evaluation - retrying")
    print(evaluation.feedback)
    return rerun(reply, message, history, evaluation.feedback)

In [ ]:
gr.ChatInterface(chat, type="messages").launch()